# 01 - Task 1: LSTM Autoencoder for Music Generation

**Course:** CSE425 / EEE474  
**Project:** Unsupervised Neural Network for Multi-Genre Music Generation  
**Author:** [Your Name Here]  
**Date:** April 2026

---

This notebook implements an LSTM Autoencoder that learns a compressed latent representation of piano-roll music sequences and uses the decoder to generate new music.

**Pipeline:**
- Encoder: $z = f_\phi(X)$
- Decoder: $\hat{X} = g_\theta(z)$
- Loss: $\mathcal{L}_{AE} = \sum_t \|x_t - \hat{x}_t\|^2$ (MSE)

**Dependencies:** Requires outputs from `00_preprocessing.ipynb` (saved in `outputs/processed/`).

## Section 1 — Setup & Data Loading

### NVIDIA GPU (why training shows `cpu`)

A plain `pip install torch` from PyPI is usually **CPU-only** — then `torch.version.cuda` is `None` and `torch.cuda.is_available()` is `False`. Your RTX 4090 is fine; PyTorch simply was not installed with CUDA libraries.

**Fix:** run the **next code cell once** with `INSTALL_CUDA_PYTORCH = True`, then **Restart Kernel**, set it back to `False`, and **Run All** from the top.

RTX 40-series works well with **CUDA 12.x** wheels (`cu124`).

---

**If training crashes with `RuntimeError: CUDA error: unknown error`** (often during `MSELoss`): on Windows, **cuDNN is disabled by default** in the setup cell below so LSTM uses stable native GPU kernels. To try faster cuDNN again, set environment variable `PYTORCH_DISABLE_CUDNN=0` before launching Jupyter/Cursor.

In [1]:
"""GPU bootstrap — run before training if `torch.version.cuda` is None."""

import shutil
import subprocess
import sys

# -----------------------------------------------------------------------------
# PyTorch from PyPI is CPU-only unless you install CUDA wheels from pytorch.org.
# Set True ONCE → run this cell → Restart Kernel → set False → Run All.
# -----------------------------------------------------------------------------
INSTALL_CUDA_PYTORCH = False


def _nvidia_smi_list():
    """Return human-readable GPU list from nvidia-smi, or an error string."""
    exe = shutil.which("nvidia-smi")
    if not exe:
        return False, "nvidia-smi not found on PATH — install NVIDIA drivers from https://www.nvidia.com/drivers/"
    try:
        proc = subprocess.run(
            [exe, "-L"],
            capture_output=True,
            text=True,
            timeout=20,
            check=False,
        )
        return True, proc.stdout.strip() or proc.stderr.strip()
    except Exception as exc:
        return False, str(exc)


def main():
    import torch

    cuda_compiled = torch.version.cuda  # None → CPU-only wheel
    cuda_ok = torch.cuda.is_available()

    print(f"torch.__version__ : {torch.__version__}")
    print(f"torch.version.cuda : {cuda_compiled}")
    print(f"torch.cuda.is_available() : {cuda_ok}")

    ok, gpu_txt = _nvidia_smi_list()
    print("\n--- nvidia-smi -L ---")
    print(gpu_txt)

    if cuda_compiled and cuda_ok:
        print("\nCUDA-enabled PyTorch is active — GPU training will work.")
        return

    if cuda_compiled and not cuda_ok:
        print(
            "\nPyTorch includes CUDA libraries but cuda.is_available() is False.\n"
            "Try updating drivers, reinstalling CUDA-enabled PyTorch, or checking env vars "
            "(CUDA_VISIBLE_DEVICES)."
        )
        return

    if not INSTALL_CUDA_PYTORCH:
        print(
            "\nCPU-only PyTorch detected.\n"
            "To use RTX 4090:\n"
            "  1) Set INSTALL_CUDA_PYTORCH = True in this cell.\n"
            "  2) Run this cell (downloads CUDA 12.4 PyTorch).\n"
            "  3) Restart Jupyter kernel.\n"
            "  4) Set INSTALL_CUDA_PYTORCH = False and Run All.\n"
            "\nManual alternative (PowerShell / terminal):\n"
            "  pip uninstall -y torch torchvision torchaudio\n"
            "  pip install torch torchvision torchaudio "
            "--index-url https://download.pytorch.org/whl/cu124"
        )
        return

    print("\nInstalling PyTorch + torchvision + torchaudio with CUDA 12.4 ...")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "torch",
            "torchvision",
            "torchaudio",
            "--index-url",
            "https://download.pytorch.org/whl/cu124",
        ]
    )
    print(
        "\n>>> Done. Restart the Jupyter kernel now (Kernel → Restart).\n"
        ">>> Then set INSTALL_CUDA_PYTORCH = False and run all cells."
    )
    sys.exit(0)


main()


torch.__version__ : 2.6.0+cu124
torch.version.cuda : 12.4
torch.cuda.is_available() : True

--- nvidia-smi -L ---
GPU 0: NVIDIA GeForce RTX 4090 (UUID: GPU-32c1f518-e945-118d-8bde-51851d937677)

CUDA-enabled PyTorch is active — GPU training will work.


In [2]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import pretty_midi

# ── Reproducibility ──────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

_use_cuda = torch.cuda.is_available()

# cuDNN's fused LSTM can trigger opaque "CUDA error: unknown error" on some Windows+NVIDIA setups.
# Disabling cuDNN keeps training on GPU via native PyTorch RNN kernels (slightly slower but stable).
# Re-enable: set env PYTORCH_DISABLE_CUDNN=0 before starting Jupyter.
_env_cudnn = os.environ.get("PYTORCH_DISABLE_CUDNN")
if _env_cudnn is None:
    DISABLE_CUDNN_FOR_STABILITY = os.name == "nt"
else:
    DISABLE_CUDNN_FOR_STABILITY = _env_cudnn.strip().lower() not in ("0", "false", "no")

if _use_cuda:
    torch.cuda.manual_seed_all(SEED)
    if DISABLE_CUDNN_FOR_STABILITY:
        torch.backends.cudnn.enabled = False
    else:
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False
else:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ── Plotting defaults ────────────────────────────────────────────────
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 150})

# ── Device (GPU when CUDA-enabled PyTorch is installed) ───────────────
device = torch.device("cuda" if _use_cuda else "cpu")

if _use_cuda:
    torch.cuda.empty_cache()
    # Warm up GPU (catches driver/context issues before the long training loop).
    w = torch.randn(256, 256, device=device, dtype=torch.float32)
    _warm_out = w @ w
    torch.cuda.synchronize()
    del w, _warm_out

print(f"Using device: {device}")
if _use_cuda:
    idx = torch.cuda.current_device()
    print(f"  GPU [{idx}]: {torch.cuda.get_device_name(idx)}")
    print(f"  CUDA (PyTorch built with): {torch.version.cuda}")
    mem_gb = torch.cuda.get_device_properties(idx).total_memory / (1024 ** 3)
    print(f"  VRAM available: {mem_gb:.1f} GB")
    print(f"  cuDNN enabled: {torch.backends.cudnn.enabled}")
else:
    print(f"  torch.version.cuda : {torch.version.cuda}  (None means CPU-only PyTorch from pip)")
    print(
        "  Training on CPU until you install CUDA-enabled PyTorch — use the GPU bootstrap "
        "cell above (INSTALL_CUDA_PYTORCH = True once), Restart Kernel, then Run All."
    )

Using device: cuda
  GPU [0]: NVIDIA GeForce RTX 4090
  CUDA (PyTorch built with): 12.4
  VRAM available: 24.0 GB


### Helper — `piano_roll_to_midi`

Converts a `(128, T)` piano-roll array back to a PrettyMIDI object (and optionally saves to disk).

In [3]:
def piano_roll_to_midi(piano_roll, fs=16, output_path=None):
    """Convert a (128, T) piano-roll matrix to a PrettyMIDI object.

    Parameters
    ----------
    piano_roll : np.ndarray
        Binary or continuous piano-roll of shape (128, T) where rows are
        MIDI pitches and columns are time frames.
    fs : int
        Frames per second (sampling rate used during preprocessing).
    output_path : str or None
        If provided, the resulting MIDI is written to this path.

    Returns
    -------
    pretty_midi.PrettyMIDI
    """
    midi = pretty_midi.PrettyMIDI(initial_tempo=120.0)
    instrument = pretty_midi.Instrument(program=0, name="Piano")

    # Binarise with a 0.5 threshold for continuous values
    piano_roll = (piano_roll > 0.5).astype(np.uint8)

    for pitch in range(128):
        active = False
        start_frame = 0
        for t in range(piano_roll.shape[1]):
            if piano_roll[pitch, t] == 1 and not active:
                active = True
                start_frame = t
            elif piano_roll[pitch, t] == 0 and active:
                active = False
                start_time = start_frame / fs
                end_time = t / fs
                note = pretty_midi.Note(
                    velocity=100, pitch=pitch,
                    start=start_time, end=end_time
                )
                instrument.notes.append(note)
        if active:
            start_time = start_frame / fs
            end_time = piano_roll.shape[1] / fs
            note = pretty_midi.Note(
                velocity=100, pitch=pitch,
                start=start_time, end=end_time
            )
            instrument.notes.append(note)

    midi.instruments.append(instrument)

    if output_path is not None:
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        midi.write(output_path)
        print(f"  Saved MIDI → {output_path}")

    return midi

### Load Pre-processed Data

In [4]:
train_sequences = np.load(os.path.join("outputs", "processed", "train_sequences.npy"))
val_sequences = np.load(os.path.join("outputs", "processed", "val_sequences.npy"))

print(f"Train sequences : {train_sequences.shape}")
print(f"Val   sequences : {val_sequences.shape}")

Train sequences : (285605, 64, 128)
Val   sequences : (34845, 64, 128)


### PyTorch Dataset & DataLoaders

In [5]:
class MusicDataset(Dataset):
    """Wraps a NumPy array of piano-roll sequences as a PyTorch Dataset."""

    def __init__(self, sequences: np.ndarray):
        """Initialise with an (N, seq_len, 128) array."""
        self.data = torch.tensor(sequences, dtype=torch.float32)

    def __len__(self):
        """Return the number of sequences."""
        return len(self.data)

    def __getitem__(self, idx):
        """Return a single sequence of shape (seq_len, 128)."""
        return self.data[idx]


BATCH_SIZE = 64

# DataLoader tuning for GPU: pin_memory speeds pinned RAM → GPU transfers.
# On Windows + Jupyter, multiprocessing workers often cause hangs → keep num_workers=0.
_pin_memory = torch.cuda.is_available()
_num_workers = min(4, max(1, (os.cpu_count() or 4) // 2))
if os.name == "nt":
    _num_workers = 0

train_dataset = MusicDataset(train_sequences)
val_dataset = MusicDataset(val_sequences)

_train_kw = dict(
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    pin_memory=_pin_memory,
    num_workers=_num_workers,
)
_val_kw = dict(
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    pin_memory=_pin_memory,
    num_workers=_num_workers,
)
if _num_workers > 0:
    _train_kw["persistent_workers"] = True
    _train_kw["prefetch_factor"] = 2
    _val_kw["persistent_workers"] = True
    _val_kw["prefetch_factor"] = 2

train_loader = DataLoader(train_dataset, **_train_kw)
val_loader = DataLoader(val_dataset, **_val_kw)

sample = train_dataset[0]
print(f"Single sample shape: {sample.shape}  (seq_len={sample.shape[0]}, input_dim={sample.shape[1]})")
print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")
print(f"DataLoader: pin_memory={_pin_memory}, num_workers={_num_workers}")

Single sample shape: torch.Size([64, 128])  (seq_len=64, input_dim=128)
Train batches: 4463  |  Val batches: 545
DataLoader: pin_memory=True, num_workers=0


---
## Section 2 — Model Architecture

| Component | Detail |
|-----------|--------|
| **Encoder** | 2-layer bidirectional LSTM (hidden=256) → Linear → latent `z` (dim=64) |
| **Decoder** | Linear expand → repeat across `seq_len` → 2-layer LSTM (hidden=256) → Linear + Sigmoid → output (batch, 64, 128) |

In [6]:
class LSTMEncoder(nn.Module):
    """Bidirectional LSTM encoder that maps (batch, seq_len, 128) → (batch, latent_dim)."""

    def __init__(self, input_dim=128, hidden_size=256, num_layers=2, latent_dim=64):
        """Initialise encoder layers.

        Parameters
        ----------
        input_dim : int
            Feature dimension per time-step (128 MIDI pitches).
        hidden_size : int
            LSTM hidden state size.
        num_layers : int
            Number of stacked LSTM layers.
        latent_dim : int
            Dimensionality of the compressed latent vector.
        """
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )
        # Forward + backward final hidden → latent
        self.fc_latent = nn.Linear(hidden_size * 2, latent_dim)

    def forward(self, x):
        """Encode a batch of sequences.

        Parameters
        ----------
        x : Tensor of shape (batch, seq_len, input_dim)

        Returns
        -------
        z : Tensor of shape (batch, latent_dim)
        """
        # h_n shape: (num_layers * 2, batch, hidden_size)
        _, (h_n, _) = self.lstm(x)

        # Take the last layer's forward and backward hidden states
        h_forward = h_n[-2]   # (batch, hidden_size)
        h_backward = h_n[-1]  # (batch, hidden_size)
        h_cat = torch.cat([h_forward, h_backward], dim=1)  # (batch, hidden_size*2)

        z = self.fc_latent(h_cat)  # (batch, latent_dim)
        return z

In [7]:
class LSTMDecoder(nn.Module):
    """LSTM decoder that maps (batch, latent_dim) → (batch, seq_len, 128)."""

    def __init__(self, latent_dim=64, hidden_size=256, num_layers=2,
                 output_dim=128, seq_len=64):
        """Initialise decoder layers.

        Parameters
        ----------
        latent_dim : int
            Size of the incoming latent vector.
        hidden_size : int
            LSTM hidden state size.
        num_layers : int
            Number of stacked LSTM layers.
        output_dim : int
            Output feature dimension (128 MIDI pitches).
        seq_len : int
            Number of time-steps to reconstruct.
        """
        super().__init__()
        self.seq_len = seq_len
        self.hidden_size = hidden_size

        self.fc_expand = nn.Linear(latent_dim, hidden_size)
        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
        )
        self.fc_out = nn.Linear(hidden_size, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, z):
        """Decode a latent vector to a piano-roll sequence.

        Parameters
        ----------
        z : Tensor of shape (batch, latent_dim)

        Returns
        -------
        x_hat : Tensor of shape (batch, seq_len, output_dim)
        """
        h = self.fc_expand(z)                        # (batch, hidden_size)
        h = h.unsqueeze(1).repeat(1, self.seq_len, 1)  # (batch, seq_len, hidden_size)

        lstm_out, _ = self.lstm(h)                   # (batch, seq_len, hidden_size)
        x_hat = self.sigmoid(self.fc_out(lstm_out))  # (batch, seq_len, output_dim)
        return x_hat

In [8]:
class LSTMAutoencoder(nn.Module):
    """Full autoencoder combining LSTMEncoder and LSTMDecoder."""

    def __init__(self, input_dim=128, hidden_size=256, num_layers=2,
                 latent_dim=64, seq_len=64):
        """Initialise encoder and decoder sub-modules."""
        super().__init__()
        self.encoder = LSTMEncoder(input_dim, hidden_size, num_layers, latent_dim)
        self.decoder = LSTMDecoder(latent_dim, hidden_size, num_layers, input_dim, seq_len)

    def forward(self, x):
        """Full forward pass: encode then decode.

        Parameters
        ----------
        x : Tensor of shape (batch, seq_len, input_dim)

        Returns
        -------
        x_hat : Tensor of shape (batch, seq_len, input_dim)
        """
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

    def encode(self, x):
        """Encode input to latent space."""
        return self.encoder(x)

    def decode(self, z):
        """Decode latent vector to reconstruction."""
        return self.decoder(z)


# ── Instantiate & summarise ───────────────────────────────────────────
SEQ_LEN = train_sequences.shape[1]    # 64
INPUT_DIM = train_sequences.shape[2]  # 128
HIDDEN_SIZE = 256
NUM_LAYERS = 2
LATENT_DIM = 64

model = LSTMAutoencoder(
    input_dim=INPUT_DIM,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    latent_dim=LATENT_DIM,
    seq_len=SEQ_LEN,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model summary:")
print(f"  Total parameters     : {total_params:,}")
print(f"  Trainable parameters : {trainable_params:,}")
print()
print(model)

Model summary:
  Total parameters     : 3,502,528
  Trainable parameters : 3,502,528

LSTMAutoencoder(
  (encoder): LSTMEncoder(
    (lstm): LSTM(128, 256, num_layers=2, batch_first=True, bidirectional=True)
    (fc_latent): Linear(in_features=512, out_features=64, bias=True)
  )
  (decoder): LSTMDecoder(
    (fc_expand): Linear(in_features=64, out_features=256, bias=True)
    (lstm): LSTM(256, 256, num_layers=2, batch_first=True)
    (fc_out): Linear(in_features=256, out_features=128, bias=True)
    (sigmoid): Sigmoid()
  )
)


---
## Section 3 — Training

- **Optimizer:** Adam (lr = 1e-3)  
- **Loss:** MSE  
- **Epochs:** 50  
- **Best weights** (lowest validation loss): `outputs/task1/best_model.pth`  
- **Resume checkpoint** (saved after each full epoch — use after crash or kernel restart): `outputs/task1/training_checkpoint.pth`  

Keep `RESUME_IF_AVAILABLE = True` to continue from the last completed epoch. Set it to `False`, or delete `training_checkpoint.pth`, to train from epoch 1 again.

*Progress within an unfinished epoch is not saved — only completed epochs are checkpointed.*


In [9]:
os.makedirs(os.path.join("outputs", "task1"), exist_ok=True)
os.makedirs(os.path.join("outputs", "plots"), exist_ok=True)

NUM_EPOCHS = 50
LR = 1e-3

# Full training state is saved after every completed epoch for crash-safe resume.
RESUME_IF_AVAILABLE = True
CHECKPOINT_PATH = os.path.join("outputs", "task1", "training_checkpoint.pth")

optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

best_val_loss = float("inf")
train_losses = []
val_losses = []
start_epoch = 1

if RESUME_IF_AVAILABLE and os.path.isfile(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    train_losses = list(ckpt["train_losses"])
    val_losses = list(ckpt["val_losses"])
    best_val_loss = float(ckpt["best_val_loss"])
    completed = int(ckpt["epoch"])
    start_epoch = completed + 1
    print(
        f"Resumed from checkpoint: epochs 1–{completed} already done; "
        f"continuing from epoch {start_epoch}/{NUM_EPOCHS}"
    )
    if ckpt.get("num_epochs") != NUM_EPOCHS or ckpt.get("lr") != LR:
        print("  (Checkpoint metadata differs from current NUM_EPOCHS/LR — continuing anyway.)")
else:
    print("Starting training from epoch 1 (no checkpoint file, or RESUME_IF_AVAILABLE=False).")

if start_epoch > NUM_EPOCHS:
    print(
        f"\nCheckpoint shows training already finished for NUM_EPOCHS={NUM_EPOCHS}. "
        f"Increase NUM_EPOCHS or delete {CHECKPOINT_PATH} to start over."
    )
else:
    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        # ── Training ──────────────────────────────────────────────────────
        model.train()
        epoch_train_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch:02d}/{NUM_EPOCHS} [train]", leave=False):
            batch = batch.to(device)
            optimizer.zero_grad()
            x_hat = model(batch)
            loss = criterion(x_hat, batch)
            loss.backward()
            optimizer.step()
            epoch_train_loss += loss.item() * batch.size(0)

        epoch_train_loss /= len(train_dataset)
        train_losses.append(epoch_train_loss)

        # ── Validation ────────────────────────────────────────────────────
        model.eval()
        epoch_val_loss = 0.0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch:02d}/{NUM_EPOCHS} [val]  ", leave=False):
                batch = batch.to(device)
                x_hat = model(batch)
                loss = criterion(x_hat, batch)
                epoch_val_loss += loss.item() * batch.size(0)

        epoch_val_loss /= len(val_dataset)
        val_losses.append(epoch_val_loss)

        # ── Best weights (lowest val loss) ──────────────────────────────
        improved = ""
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(), os.path.join("outputs", "task1", "best_model.pth"))
            improved = " ✓ saved best"

        # ── Full resume checkpoint (end of epoch) ─────────────────────────
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_losses": train_losses,
                "val_losses": val_losses,
                "best_val_loss": best_val_loss,
                "num_epochs": NUM_EPOCHS,
                "lr": LR,
            },
            CHECKPOINT_PATH,
        )

        print(
            f"Epoch {epoch:02d}/{NUM_EPOCHS}  |  "
            f"Train Loss: {epoch_train_loss:.6f}  |  "
            f"Val Loss: {epoch_val_loss:.6f}{improved}"
        )

    print("\n=== Training complete ===")
    print(f"Best validation loss: {best_val_loss:.6f}")



Epoch 01/50 [train]:   0%|          | 0/4463 [00:00<?, ?it/s]

Epoch 01/50 [val]  :   0%|          | 0/545 [00:00<?, ?it/s]

Epoch 01/50  |  Train Loss: 0.011595  |  Val Loss: 0.011499 ✓ saved best


Epoch 02/50 [train]:   0%|          | 0/4463 [00:00<?, ?it/s]

Epoch 02/50 [val]  :   0%|          | 0/545 [00:00<?, ?it/s]

...

Epoch 05/50 [train]:   0%|          | 0/4463 [00:00<?, ?it/s]

Epoch 05/50 [val]  :   0%|          | 0/545 [00:00<?, ?it/s]

Epoch 05/50  |  Train Loss: 0.011256  |  Val Loss: 0.011494 ✓ saved best


Epoch 06/50 [train]:   0%|          | 0/4463 [00:00<?, ?it/s]

RuntimeError: CUDA error: unknown error
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


---
## Section 4 — Visualizations

### 4.1 Training & Validation Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
epochs_range = range(1, len(train_losses) + 1)  # resumed / partial runs
ax.plot(epochs_range, train_losses, label="Train Loss", linewidth=1.5)
ax.plot(epochs_range, val_losses, label="Validation Loss", linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Task 1 — LSTM Autoencoder Loss Curve")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join("outputs", "plots", "task1_loss_curve.png"), dpi=150)
plt.show()
print("Saved → outputs/plots/task1_loss_curve.png")


### 4.2 Piano-Roll Reconstruction Comparison

In [ ]:
# Load best model weights for evaluation
model.load_state_dict(torch.load(os.path.join("outputs", "task1", "best_model.pth"),
                                 map_location=device))
model.eval()

# Pick one validation sample
sample_original = val_dataset[0].unsqueeze(0).to(device)  # (1, seq_len, 128)
with torch.no_grad():
    sample_reconstructed = model(sample_original)

orig_np = sample_original.squeeze(0).cpu().numpy().T       # (128, seq_len)
recon_np = sample_reconstructed.squeeze(0).cpu().numpy().T  # (128, seq_len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(orig_np, aspect="auto", origin="lower", cmap="magma")
axes[0].set_title("Original Piano Roll")
axes[0].set_xlabel("Time Step")
axes[0].set_ylabel("MIDI Pitch")

im = axes[1].imshow(recon_np, aspect="auto", origin="lower", cmap="magma")
axes[1].set_title("Reconstructed Piano Roll")
axes[1].set_xlabel("Time Step")
axes[1].set_ylabel("MIDI Pitch")

fig.colorbar(im, ax=axes, shrink=0.6, label="Activation")
fig.suptitle("Task 1 — Original vs Reconstructed", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join("outputs", "plots", "task1_reconstruction.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/plots/task1_reconstruction.png")

---
## Section 5 — Music Generation

Generate 5 new pieces by sampling from the latent space $z \sim \mathcal{N}(0, I)$ and decoding.

In [ ]:
MIDI_OUTPUT_DIR = os.path.join("outputs", "task1", "generated_midis")
os.makedirs(MIDI_OUTPUT_DIR, exist_ok=True)

NUM_SAMPLES = 5
generated_rolls = []

model.eval()
with torch.no_grad():
    for i in range(1, NUM_SAMPLES + 1):
        z = torch.randn(1, LATENT_DIM, device=device)  # (1, 64)
        x_hat = model.decode(z)                         # (1, seq_len, 128)

        piano_roll = x_hat.squeeze(0).cpu().numpy().T   # (128, seq_len)
        generated_rolls.append(piano_roll)

        midi_path = os.path.join(MIDI_OUTPUT_DIR, f"generated_{i}.mid")
        piano_roll_to_midi(piano_roll, fs=16, output_path=midi_path)

print(f"\nGenerated {NUM_SAMPLES} MIDI files in {MIDI_OUTPUT_DIR}/")

---
## Section 6 — Evaluation Metrics

For each generated sample we compute:

| Metric | Formula |
|--------|---------|
| **Rhythm Diversity** | $D_{\text{rhythm}} = \frac{\text{\# unique durations}}{\text{\# total notes}}$ |
| **Repetition Ratio** | $R = \frac{\text{\# repeated patterns}}{\text{\# total patterns}}$ (sliding window of 4 bars) |
| **Pitch Entropy** | Shannon entropy of the 12-bin pitch-class histogram |

In [ ]:
def compute_rhythm_diversity(midi_obj):
    """Compute rhythm diversity as the ratio of unique note durations to total notes.

    Parameters
    ----------
    midi_obj : pretty_midi.PrettyMIDI

    Returns
    -------
    float
        Rhythm diversity score in [0, 1]. Returns 0 if no notes exist.
    """
    notes = [n for inst in midi_obj.instruments for n in inst.notes]
    if len(notes) == 0:
        return 0.0
    durations = [round(n.end - n.start, 4) for n in notes]
    return len(set(durations)) / len(durations)


def compute_repetition_ratio(piano_roll, window_size=4):
    """Compute the repetition ratio using a sliding window over the piano roll.

    Parameters
    ----------
    piano_roll : np.ndarray
        Shape (128, T) binary piano roll.
    window_size : int
        Number of time-step columns per pattern window.

    Returns
    -------
    float
        Ratio of repeated patterns to total patterns.
    """
    binary_roll = (piano_roll > 0.5).astype(np.uint8)
    num_steps = binary_roll.shape[1]
    if num_steps < window_size:
        return 0.0

    pattern_hashes = []
    for t in range(num_steps - window_size + 1):
        window = binary_roll[:, t:t + window_size]
        pattern_hashes.append(hash(window.tobytes()))

    total_patterns = len(pattern_hashes)
    unique_patterns = len(set(pattern_hashes))
    repeated = total_patterns - unique_patterns
    return repeated / total_patterns


def compute_pitch_entropy(midi_obj):
    """Compute Shannon entropy of the 12-bin pitch-class histogram.

    Parameters
    ----------
    midi_obj : pretty_midi.PrettyMIDI

    Returns
    -------
    float
        Entropy in bits. Returns 0 if no notes exist.
    """
    notes = [n for inst in midi_obj.instruments for n in inst.notes]
    if len(notes) == 0:
        return 0.0
    pitch_classes = [n.pitch % 12 for n in notes]
    hist, _ = np.histogram(pitch_classes, bins=12, range=(0, 12))
    hist = hist / hist.sum()
    hist = hist[hist > 0]
    entropy = -np.sum(hist * np.log2(hist))
    return entropy

In [ ]:
metrics_records = []

for i in range(NUM_SAMPLES):
    midi_path = os.path.join(MIDI_OUTPUT_DIR, f"generated_{i + 1}.mid")
    midi_obj = pretty_midi.PrettyMIDI(midi_path)
    roll = generated_rolls[i]

    rd = compute_rhythm_diversity(midi_obj)
    rr = compute_repetition_ratio(roll, window_size=4)
    pe = compute_pitch_entropy(midi_obj)

    metrics_records.append({
        "sample_id": i + 1,
        "rhythm_diversity": round(rd, 4),
        "repetition_ratio": round(rr, 4),
        "pitch_entropy": round(pe, 4),
    })

metrics_df = pd.DataFrame(metrics_records)
print("=== Generated Sample Metrics ===")
print(metrics_df.to_string(index=False))

metrics_csv_path = os.path.join("outputs", "task1", "metrics.csv")
metrics_df.to_csv(metrics_csv_path, index=False)
print(f"\nSaved → {metrics_csv_path}")

---

**End of Notebook 01 — Task 1: LSTM Autoencoder**

Outputs produced:
- `outputs/task1/best_model.pth` — best model checkpoint
- `outputs/plots/task1_loss_curve.png` — loss curve
- `outputs/plots/task1_reconstruction.png` — reconstruction comparison
- `outputs/task1/generated_midis/generated_1.mid` … `generated_5.mid` — generated music
- `outputs/task1/metrics.csv` — evaluation metrics